# Enterprise Data Agent Workshop — Complete

**Build a memory-aware enterprise data agent on Oracle AI Database 26ai.**

The Codespace has already done all the Oracle setup for you (`app/scripts/bootstrap.py` + `app/scripts/seed.py` + `app/scripts/setup_advanced.py` ran on first launch). This notebook focuses on the **Python harness** — the loop on top of the LLM that turns a tokens-in/tokens-out model into an agent.

| What's pre-built (already in the Codespace Environment) | What you'll build in this notebook |
|---|---|
| `AGENT` user, `vector_memory_size`, `pga_aggregate_limit` | The `OracleONNXEmbedder` + OAMP wiring |
| `ALL_MINILM_L12_V2` embedder + `RERANKER_ONNX` reranker | The scanner that turns catalog views into memories |
| `SUPPLYCHAIN` schema (carriers, ports, vessels, voyages, …) | Retrieval (vector + hybrid via RRF) |
| `voyage_dv` + `vessel_dv` duality views | The `@register` decorator + tool registry |
| `eda_memory_text_idx` (Oracle Text on the OAMP body) | `agent_turn` — the heart of the harness |
| | Tool-output offload + retrieval |
| Skillbox seeded from [`oracle/skills`](https://github.com/oracle/skills) | |

![Enterprise Data Agent — Oracle-Native Architecture](../images/cover-oracle-native-arch.png)

> 📖 **Workshop guide:** [`README.md`](../README.md). Each Part has its own deep-dive in [`docs/`](../docs/).
>
> 🛠️ This is the **student** notebook.
>
> Each TODO has a hard-stop assert below it — fix the TODO before moving on. 
>
> The complete solution is in `notebook_complete.ipynb`; the version that includes Oracle DDL is > in `notebook_complete_with_setup_code.ipynb`.

## What is an "agent harness"?

A large-language model on its own is a pure function: tokens in, tokens out. It can't:

- Remember anything between calls.
- Execute code or SQL.
- See the world (no filesystem, no database, no internet).
- Set up its own environment.

Everything else — every piece of code, configuration, and execution logic that is *not the model* — is the **harness**:

```
Agent  =  Model  +  Harness
```

This notebook builds the harness for an **enterprise data agent**: the loop on top of an LLM that gives it persistent memory, tool dispatch over a vector-indexed registry, retrieval that combines semantic similarity with exact-token precision, a JavaScript sandbox for deterministic math, and identity-aware authorization enforced by the database kernel — not by your application code.

### How we build it — Lego by Lego

Every Part adds one block. By the end of Part 11 you have a working agent.

| Part | What it adds | Building on |
|---|---|---|
| 1 | Imports, an Oracle connection, an LLM chat helper | (foundation) |
| 2 | Long-term memory + a scanner that mines Oracle's catalog into memories | the connection from Part 1 |
| 3 | Retrieval — cosine + rerank + hybrid Reciprocal Rank Fusion | the memories from Part 2 |
| 4 | DBFS scratchpad — a real filesystem inside Oracle | (independent primitive) |
| 5 | Oracle MLE — a JavaScript sandbox for deterministic compute | (independent primitive) |
| 6 | Tools (`@register`) + skills (markdown playbooks) | retrieval from Part 3 |
| 7 | The agent loop — `agent_turn` orchestrates everything above | every block before it |
| 9 | JSON Relational Duality Views — same row, JSON shape | the agent loop |
| 11 | Tool-output offload — preview inline, full bytes recoverable | the agent loop + memory |

If you get lost, stop and re-read the table. Every cell below is *adding one more block* to this stack.

## At a glance — the 9 TODOs and the running app

**9 hands-on coding TODOs across 11 parts.** Every "true setup" task (Oracle DDL, seed data, ONNX models) was run by the Codespace before you opened this notebook — so you focus on the harness, not the plumbing.

Each TODO has a **hard-stop assert checkpoint** right below it. Skip or break a TODO and the next cell raises an `AssertionError` so you can't accidentally barrel forward with broken state.

| Part | Topic | TODO(s) |
|---|---|---|
| 1 | Setup & connectivity | — |
| 2 | Long-term memory with OAMP | **TODO 1** — `_scan_tables` |
| 3 | Retrieval (vector + hybrid RRF) | **TODO 2** — `retrieve_knowledge`<br>**TODO 3** — `hybrid_rrf_search_memories` |
| 4 | DBFS scratchpad | — |
| 5 | Oracle MLE compute sandbox | — |
| 6 | Tools & skills | **TODO 4** — register `tool_run_sql` |
| 7 | The agent loop | **TODO 5** — `agent_turn` |
| 9 | JSON Relational Duality Views | **TODO 7** — `tool_get_document` |
| 11 | Tool-output offload | **TODO 8** — `log_tool`<br>**TODO 9** — `tool_fetch_tool_output` |

> Full checklist: [`docs/TODO-checklist.md`](../docs/TODO-checklist.md).

### From notebook concept → running application

**The same harness you build here is already running as a Flask + React app** against the *same* Oracle, *same* OAMP store, *same* `toolbox` and `skillbox`. The notebook teaches the concept; the app shows it deployed.

> **Open the running app now:** [http://localhost:3000](http://localhost:3000) — the Codespace auto-forwards port 3000 and opens a preview when the UI is ready. (If it didn't open, check the **PORTS** tab in VS Code.)
>
> Architecture: [`app/README.md`](../app/README.md). Bootstrap source: `app/scripts/bootstrap.py` / `seed.py` / `setup_advanced.py`.

# Part 1 — Setup

> 📖 **Guide:** [`docs/part-1-setup.md`](../docs/part-1-setup.md)

The Codespace ran `app/scripts/bootstrap.py`, `app/scripts/seed.py`, and `app/scripts/setup_advanced.py` on first launch. That means Oracle already has:

- The `AGENT` user with all required grants.
- `vector_memory_size = 512M` and `pga_aggregate_limit = 4G`.
- The in-DB ONNX embedder (`ALL_MINILM_L12_V2`) and reranker (`RERANKER_ONNX`).
- The `SUPPLYCHAIN` schema seeded with ~580 rows.
- `voyage_dv` + `vessel_dv` JSON Relational Duality Views.
- The Oracle Text index on the OAMP memory body.
- The skillbox populated from [`oracle/skills`](https://github.com/oracle/skills).

You don't run any DDL in this notebook. The cells below just open a Python connection to the already-bootstrapped Oracle and wire up the chat LLM.

## 1.1 Imports + credentials

We import only what the notebook itself needs. The chat LLM provider is picked via `LLM_PROVIDER` (`openai` or `oci`); both speak the OpenAI wire protocol via the `openai` SDK.

In [ ]:
import os, time, json, re, hashlib, uuid, getpass, warnings, concurrent.futures
import numpy as np
import oracledb
from dataclasses import dataclass
from openai import OpenAI

warnings.filterwarnings("ignore", category=UserWarning, module="oracleagentmemory")

LLM_PROVIDER = os.environ.setdefault("LLM_PROVIDER", "oci").lower()
assert LLM_PROVIDER in ("openai", "oci"), "LLM_PROVIDER must be 'openai' or 'oci'"

if LLM_PROVIDER == "oci":
    if not os.environ.get("OCI_GENAI_API_KEY"):
        os.environ["OCI_GENAI_API_KEY"] = getpass.getpass("OCI GenAI API key: ")
    os.environ.setdefault("OCI_GENAI_ENDPOINT",
                          "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com")
    os.environ.setdefault("LLM_MODEL", "xai.grok-4-1-fast-reasoning")
else:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    os.environ.setdefault("LLM_MODEL", "gpt-5.5")

print(f"LLM: {LLM_PROVIDER}/{os.environ['LLM_MODEL']}")

## 1.2 Connect to the AGENT user

A thin retry wrapper around `oracledb.connect`. Oracle is already running; we just open a session as `AGENT`.

### What is a "user" in Oracle?

In Oracle, a **user** is *both* a login identity *and* a schema (the namespace of tables, views, indexes, procedures it owns). When you `CREATE USER agent IDENTIFIED BY ...`, you are simultaneously creating:

- A login (someone can authenticate as `agent` over SQL*Net).
- An empty schema named `AGENT` where any object that user creates lives (e.g. `AGENT.eda_onnx_memory`).

The Codespace already created two database users for the workshop:

| Oracle user | Owns | Used by |
|---|---|---|
| `SYS` | the entire database (super-user) | one-time bootstrap only |
| `AGENT` | the harness's own state — OAMP memory tables, the `toolbox`/`skillbox` vector-indexed tables, the DBFS scratchpad | every line of code below |
| `SUPPLYCHAIN` | the *business* data — `carriers`, `vessels`, `voyages`, `containers`, `cargo_items` | the agent reads from here via `tool_run_sql` (Part 6) |

The split is the trust boundary: if a hostile prompt got the agent to issue `DROP TABLE`, it could only drop something **`AGENT`** owns. The business data is safe by grants alone, no Python checks required.

### What does the harness mean by "user" in other contexts?

Three meanings, easy to mix up — keep them straight:

| Term | What it is | Where in the harness |
|---|---|---|
| **Oracle user / schema** | A database login + namespace. `AGENT`, `SUPPLYCHAIN`, etc. | This cell's `agent_conn` |
| **OAMP `user_id`** | A *scoping ID* tagged on every memory. Marks who the operator is in the workshop, distinct from the database user. | Part 2: `USER_ID = "enterprise-operator"` |

All three are wired up below — just know that "user" overloads in this codebase and the table above is the legend.

In [ ]:
SYS_DSN    = "localhost:1521/FREEPDB1"
AGENT_USER = "AGENT"
AGENT_PASS = "AgentPwd_2025"
DEMO_USER  = "SUPPLYCHAIN"

def connect(user, password, dsn, mode=None, retries=5):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            kwargs = dict(user=user, password=password, dsn=dsn)
            if mode is not None:
                kwargs["mode"] = mode
            conn = oracledb.connect(**kwargs)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE rownum = 1")
                print(f"connected as {user}@{dsn} — {cur.fetchone()[0]}")
            return conn
        except Exception as e:
            last_err = e
            time.sleep(3)
    raise RuntimeError(f"could not connect to {dsn}: {last_err}")


agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)

## 1.3 Chat LLM client

The standard `openai` SDK, pointed at OpenAI directly or at OCI GenAI's OpenAI-compatible endpoint depending on `LLM_PROVIDER`.

In [ ]:
LLM_MODEL = os.environ["LLM_MODEL"]

if LLM_PROVIDER == "oci":
    llm = OpenAI(base_url=os.environ["OCI_GENAI_ENDPOINT"],
                 api_key=os.environ["OCI_GENAI_API_KEY"])
else:
    llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def chat(messages, tools=None, model=LLM_MODEL, max_retries=3):
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    delay = 2.0
    for attempt in range(max_retries + 1):
        try:
            return llm.chat.completions.create(**kwargs)
        except Exception as e:
            status = getattr(e, "status_code", None) or 0
            if status == 429 and attempt < max_retries:
                time.sleep(delay); delay *= 2
            else:
                raise


resp = chat([
    {"role": "system", "content": "Be terse."},
    {"role": "user",   "content": "Say 'pong'."},
])
print(f"[{LLM_PROVIDER}/{LLM_MODEL}]", resp.choices[0].message.content)

# Part 2 — Long-Term Memory with OAMP

> 📖 **Guide:** [`docs/part-2-oamp-memory.md`](../docs/part-2-oamp-memory.md)

> 🔧 **TODO in this part:** **TODO 1** — `_scan_tables`

*Block 2 of 11.* Part 1 gave us a Python connection (`agent_conn`) and a chat client (`chat`). Part 2 lays the first foundation stone of the harness: **a place to store what the agent learns.** If the agent is going to know things across turns — schema facts, user corrections, past tool outputs — those things have to live somewhere durable. That somewhere is OAMP.


The long-term store is the [Oracle AI Agent Memory Package (OAMP)](https://www.oracle.com/database/ai-agent-memory/). It owns three primitives:

| OAMP primitive | What it stores |
|---|---|
| `memory` | Durable facts — scanned schema entries, corrections, tool outputs |
| `thread` | A conversation. Holds messages, exposes a context card |
| `context_card` | Compact, query-relevant block of memories + recent turns |

We hand it a Python connection and an embedder, and let it own the DDL. The tables (`eda_onnx_memory`, `eda_onnx_thread`, `eda_onnx_record_chunks`) are auto-created on first use.

## 2.1 In-DB ONNX embedder for OAMP

### What is an embedder?

An **embedder** is a model that turns text into a fixed-length vector of floats. Two texts with similar *meaning* land near each other in that vector space, regardless of whether they share any words. That's what lets the agent search by meaning instead of by exact match.

We use `ALL_MINILM_L12_V2`, a 384-dimensional sentence embedder, **loaded into the database** via `DBMS_VECTOR.LOAD_ONNX_MODEL` (the Codespace did this for you in `app/scripts/bootstrap.py`). Calling it is a SQL function: `VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :t AS DATA)`.

### Why in-database?

OAMP needs an embedder for every `add_memory` call (write path) and every `search` call (read path). Most stacks make these network calls to OpenAI's `text-embedding-3-small` or similar. We don't — every embed call is a SQL statement on `agent_conn`. **Zero network calls for embedding. Same trust boundary as your data. No external dependency.**

The wrapper below adapts that SQL function to OAMP's `IEmbedder` Python interface.

In [ ]:
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder

ONNX_EMBED_MODEL = "ALL_MINILM_L12_V2"
ONNX_EMBED_DIM   = 384


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name=ONNX_EMBED_MODEL, dim=ONNX_EMBED_DIM):
        self._conn = conn
        self._model = model_name
        self._dim = dim

    def embed(self, texts, *, is_query=False):
        out = np.zeros((len(texts), self._dim), dtype=np.float32)
        sql = f"SELECT VECTOR_EMBEDDING({self._model} USING :t AS DATA) FROM dual"
        with self._conn.cursor() as cur:
            for i, t in enumerate(texts):
                cur.execute(sql, t=t)
                out[i] = np.asarray(list(cur.fetchone()[0]), dtype=np.float32)
        return out

    async def embed_async(self, texts, *, is_query=False):
        return self.embed(texts, is_query=is_query)

## 2.2 Why `Llm("openai/...")` even when `LLM_PROVIDER=oci`

OAMP uses LiteLLM under the hood. LiteLLM's `Llm("openai/<model>", api_base=..., api_key=...)` means *"use the OpenAI-compatible HTTP client, pointed at this base URL"* — **not** *"call OpenAI's servers"*.

OCI GenAI exposes an OpenAI-wire-protocol endpoint. So when `LLM_PROVIDER=oci`, we pass `openai/xai.grok-4-…` as the model id and override `api_base` to OCI's endpoint. The `openai/` prefix is the *client choice*; the `api_base` is the *destination*. No traffic actually goes to OpenAI.

In [ ]:
if LLM_PROVIDER == "oci":
    extraction_llm = Llm(
        f"openai/{LLM_MODEL}",                              # client = openai-compatible
        api_base=os.environ["OCI_GENAI_ENDPOINT"],          # destination = OCI GenAI
        api_key=os.environ["OCI_GENAI_API_KEY"],
    )
else:
    extraction_llm = Llm(LLM_MODEL)                          # bare = real OpenAI


memory_client = OracleAgentMemory(
    connection=agent_conn,
    embedder=OracleONNXEmbedder(agent_conn),
    llm=extraction_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
    table_name_prefix="eda_onnx_",
)


USER_ID  = "enterprise-operator"
AGENT_ID = "enterprise-data-agent"
for register_fn, eid, info in [
    (memory_client.add_user,  USER_ID,  "Operator querying the enterprise database in natural language."),
    (memory_client.add_agent, AGENT_ID, "Data agent grounded in scanned schema metadata."),
]:
    try:
        register_fn(eid, info)
    except ValueError as e:
        if "already exists" not in str(e):
            raise

print(f"OAMP wired (in-DB ONNX embedder + {LLM_PROVIDER} extraction LLM); user/agent registered.")

## 2.3 `Fact` dataclass and scanner helpers

### What is "scanning" and why does it matter?

When a human analyst joins your team, they don't ask the LLM to guess your schema — they spend a week reading `ALL_TABLES`, hovering over `COMMENT ON COLUMN`s, looking at `V$SQL` to see how the database is actually used, and slowly building a mental model of *which table has the voyage manifest*, *what `capacity_teu` is measured in*, *what edges to join through*.

A **scan** turns that learning into a one-shot offline job: read Oracle's catalog views, convert each entry into a natural-language sentence (one per table, one per column, one per foreign key, one per observed query), embed each sentence into a vector, and store it in OAMP. The agent now has the same mental model — searchable by meaning.

Without scanning the agent would:

- Hallucinate tables and columns that don't exist.
- Forget the unit of `unit_value_cents` (it's *cents*, not dollars — the kind of fact only a `COMMENT ON COLUMN` captures).
- Not know which foreign keys to join through.
- Confidently write SQL against tables it has no evidence exist.

With scanning, the agent answers *"which table has the voyage manifest?"* by retrieving the embedded fact *"Table SUPPLYCHAIN.CONTAINERS. Documented purpose: Container manifest: which container is on which voyage..."* and grounds every later SQL in real catalog evidence.

### The four sources we mine

| Source | Catalog views | Fact `kind` | Why we mine it |
|---|---|---|---|
| Structural | `ALL_TABLES`, `ALL_TAB_COMMENTS` | `table` | *What shapes exist.* The base inventory. |
| Columns | `ALL_TAB_COLUMNS`, `ALL_COL_COMMENTS` | `column` | *What each field means.* `COMMENT ON COLUMN` text is gold. |
| Relational | `ALL_CONSTRAINTS`, `ALL_CONS_COLUMNS` | `relationship` | *How tables relate.* Foreign-key edges → join paths. |
| Workload | `V$SQL` | `query_pattern` | *How the database is actually used.* The "query inference" layer. |

Each helper takes `(conn, owner)` and returns `list[Fact]`. The `Fact` dataclass is the in-memory shape; `write_facts` (later) turns each Fact into an OAMP memory row with an embedding.

> **Why store facts as text with embeddings, not as structured rows?** Because the agent retrieves by *meaning*, not by primary key. When the user asks *"which table has the voyage manifest?"* we want a cosine search over embedded descriptions to surface `SUPPLYCHAIN.CONTAINERS`, not a JOIN through four catalog views. The structured catalog is still one `SELECT` away if the agent needs it — but the semantic layer on top is what makes natural-language queries work.

In [ ]:
@dataclass
class Fact:
    kind: str        # 'table' | 'column' | 'relationship' | 'query_pattern'
    subject: str     # e.g. 'SUPPLYCHAIN.VESSELS'
    body: str        # natural-language sentence the embedder will read
    metadata: dict



## 2.4 Implement `_scan_tables`

Mine `ALL_TABLES + ALL_TAB_COMMENTS` and emit one `Fact(kind="table")` per table. Build the `body` conditionally: skip parts where the column is `None`.

> 📖 **See:** [Part 2 guide → TODO 1](../docs/part-2-oamp-memory.md#todo-4-implement-_scan_tables)


In [ ]:
# TODO 1 — implement _scan_tables(conn, owner) -> list[Fact]
# Query: SELECT t.table_name, tc.comments, t.num_rows, t.last_analyzed
#          FROM all_tables t LEFT JOIN all_tab_comments tc USING(owner, table_name)
#         WHERE t.owner = :owner ORDER BY t.table_name
# For each row, build a body string and append a Fact(kind="table", subject=f"{owner}.{table}", ...).
# 📖 docs/part-2-oamp-memory.md

def _scan_tables(conn, owner):
    sql = (
        "SELECT t.table_name, tc.comments, t.num_rows, t.last_analyzed "
        "  FROM all_tables t "
        "  LEFT JOIN all_tab_comments tc "
        "    ON tc.owner = t.owner AND tc.table_name = t.table_name "
        " WHERE t.owner = :owner "
        " ORDER BY t.table_name"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, comment, num_rows, last_analyzed in cur:
            body_parts = [f"Table {owner}.{table}."]
            if comment:
                body_parts.append(f"Documented purpose: {comment}")
            if num_rows is not None:
                body_parts.append(f"Approximate row count: {num_rows:,}.")
            if last_analyzed:
                body_parts.append(f"Statistics last gathered at {last_analyzed}.")
            facts.append(Fact("table", f"{owner}.{table}", " ".join(body_parts),
                              {"owner": owner, "table": table,
                               "num_rows": num_rows, "has_comment": bool(comment)}))
    return facts


In [ ]:
def _scan_columns(conn, owner):
    sql = (
        "SELECT c.table_name, c.column_name, c.data_type, c.data_length, "
        "       c.nullable, cc.comments "
        "  FROM all_tab_columns c "
        "  LEFT JOIN all_col_comments cc "
        "    ON cc.owner = c.owner AND cc.table_name = c.table_name "
        "   AND cc.column_name = c.column_name "
        " WHERE c.owner = :owner "
        " ORDER BY c.table_name, c.column_id"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, col, dtype, dlen, nullable, comment in cur:
            dtype_str = dtype + (f"({dlen})" if dlen and dtype in ("VARCHAR2", "CHAR") else "")
            nullstr = "nullable" if nullable == "Y" else "NOT NULL"
            body = f"Column {owner}.{table}.{col} of type {dtype_str} ({nullstr})."
            if comment:
                body += f" Meaning: {comment}"
            facts.append(Fact("column", f"{owner}.{table}.{col}", body,
                              {"owner": owner, "table": table, "column": col,
                               "data_type": dtype, "nullable": nullable == "Y"}))
    return facts


In [ ]:
def _scan_relationships(conn, owner):
    sql = (
        "SELECT c.constraint_name, c.table_name, acc.column_name, "
        "       rc.table_name AS r_table, rcc.column_name AS r_column "
        "  FROM all_constraints c "
        "  JOIN all_cons_columns acc "
        "    ON acc.owner = c.owner AND acc.constraint_name = c.constraint_name "
        "  JOIN all_constraints rc "
        "    ON rc.owner = c.r_owner AND rc.constraint_name = c.r_constraint_name "
        "  JOIN all_cons_columns rcc "
        "    ON rcc.owner = rc.owner AND rcc.constraint_name = rc.constraint_name "
        "   AND rcc.position = acc.position "
        " WHERE c.owner = :owner AND c.constraint_type = 'R'"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for cname, tbl, col, r_tbl, r_col in cur:
            facts.append(Fact("relationship",
                              f"{owner}.{tbl}.{col}->{owner}.{r_tbl}.{r_col}",
                              f"Foreign key {cname}: {owner}.{tbl}.{col} references "
                              f"{owner}.{r_tbl}.{r_col}. Use this edge when joining the two tables.",
                              {"owner": owner, "child": tbl, "child_col": col,
                               "parent": r_tbl, "parent_col": r_col}))
    return facts

In [ ]:
def _scan_workload(conn, owner, limit=50):
    sql = (
        "SELECT sql_id, sql_fulltext, executions, rows_processed "
        "  FROM v$sql "
        " WHERE parsing_schema_name = :owner "
        "   AND command_type IN (3, 6, 7, 189) "
        "   AND rownum <= :lim "
        " ORDER BY executions DESC NULLS LAST"
    )
    facts = []
    with conn.cursor() as cur:
        try:
            cur.execute(sql, owner=owner.upper(), lim=limit)
            for sql_id, text, execs, rows_proc in cur:
                if text is None:
                    continue
                stmt = (text.read() if hasattr(text, "read") else str(text)).strip()
                if len(stmt) > 2000:
                    stmt = stmt[:2000] + " /* ...truncated */"
                facts.append(Fact("query_pattern", f"v$sql:{sql_id}",
                                  f"A SQL statement observed in the workload for {owner} "
                                  f"(executed {execs or 0} times, {rows_proc or 0} rows): {stmt}",
                                  {"owner": owner, "sql_id": sql_id,
                                   "executions": execs, "rows_processed": rows_proc}))
        except oracledb.DatabaseError as e:
            print(f"  (workload scan skipped: {e})")
    return facts


print("scanner helpers ready: _scan_columns, _scan_relationships, _scan_workload")
print("(_scan_tables is your TODO 1 — defined in the next cell)")

In [ ]:
# Hard-stop checkpoint: TODO 1 must work before later parts run.
_facts = _scan_tables(agent_conn, DEMO_USER)
assert _facts, "❌ TODO 1: _scan_tables returned no facts. SUPPLYCHAIN should have 7 tables."
assert isinstance(_facts[0], Fact), "❌ TODO 1: must return Fact objects, not dicts/tuples."
assert _facts[0].kind == "table", "❌ TODO 1: Facts must have kind='table'."
print(f"✅ TODO 1 passed — _scan_tables found {len(_facts)} tables in {DEMO_USER}.")

## 2.5 `scan_schema` and `write_facts`

`scan_schema` orchestrates all four scanners. 

`write_facts` does the idempotent upsert into OAMP — each `(kind, subject)` pair has at most one current memory row, deduped by `body_hash`.

In [ ]:
def scan_schema(conn, owner):
    facts = []
    facts += _scan_tables(conn, owner)
    facts += _scan_columns(conn, owner)
    facts += _scan_relationships(conn, owner)
    facts += _scan_workload(conn, owner)
    return facts


def _hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:32]


def write_facts(facts, scan_id=None):
    new = updated = skipped = 0
    scan_id = scan_id or str(uuid.uuid4())
    store = memory_client._store

    for f in facts:
        body_hash = _hash(f.body)
        existing = store.list(
            "memory",
            user_id=USER_ID, agent_id=AGENT_ID,
            metadata_filter={"kind": f.kind, "subject": f.subject},
            limit=1,
        )
        meta = {**f.metadata, "kind": f.kind, "subject": f.subject,
                "body_hash": body_hash, "scan_id": scan_id}

        if not existing:
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            new += 1
        elif (existing[0].metadata or {}).get("body_hash") == body_hash:
            skipped += 1
        else:
            memory_client.delete_memory(existing[0].id)
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            updated += 1
    return new, updated, skipped


def run_scan(conn, owner):
    facts = scan_schema(conn, owner)
    new, updated, skipped = write_facts(facts)
    return {"facts_total": len(facts), "new": new, "updated": updated, "skipped": skipped}


summary = run_scan(agent_conn, DEMO_USER)
print(json.dumps(summary, indent=2))

# Part 3 — Retrieval

> 📖 **Guide:** [`docs/part-3-retrieval.md`](../docs/part-3-retrieval.md)

> 🔧 **TODOs in this part (2):** **TODO 2** — `retrieve_knowledge`; **TODO 3** — `hybrid_rrf_search_memories`

*Block 3 of 11.* Part 2 gave us a place to *store* memories. Part 3 makes them *retrievable* by meaning, by exact token, and by both at once. Storage without retrieval is a write-only log; retrieval is what makes memory useful to the agent on every turn.


Two retrieval surfaces:

| Layer | What it gives you | When to use |
|---|---|---|
| Vector + rerank | Cosine similarity + cross-encoder rescoring | "Where do we record vessel current positions?" |
| Hybrid (vector + Oracle Text + RRF) | Both, fused via Reciprocal Rank Fusion | "TEU 20-foot equivalent" — concept + precise acronym |

Both run server-side. No round-trips to a separate vector DB.

## 3.1 `rerank()` — cross-encoder rescoring

### Why two stages?

Cosine similarity is fast but coarse — it scores the query and each document *independently* (one vector for the query, one per document, then cosine). It misses cases where the right answer depends on how the query and document interact.

A **cross-encoder reranker** reads the query *and* a candidate document *together*, so it can rescore based on actual relevance. It's much more accurate per pair, but much slower per call. The combined trick:

1. **Stage 1 — cosine, fast.** Pull top-`k*4` candidates from the HNSW index. Cheap.
2. **Stage 2 — reranker, slow.** Reorder *only* those candidates by cross-encoder score.

You get the recall of vector search with the precision of a cross-encoder, at a fraction of the latency of running the cross-encoder over everything.

### Inside Oracle

Our reranker is also loaded into the database (via the same `DBMS_VECTOR.LOAD_ONNX_MODEL` call, with different metadata). It's invoked as a standard SQL `PREDICTION(RERANKER_ONNX USING :q AS DATA1, :doc AS DATA2)`. The Python wrapper below ships the candidates server-side as a `JSON_TABLE`, scores them all in one round trip, and returns the top-k.

If no reranker is loaded, the helper gracefully falls through to cosine ordering — call sites stay the same.

In [ ]:
RERANKER_MODEL = "RERANKER_ONNX"


def _reranker_loaded():
    with agent_conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
                    m=RERANKER_MODEL)
        return cur.fetchone()[0] > 0


def rerank(query, candidates, top_k=5, content_key="body"):
    if not candidates:
        return candidates
    if not _reranker_loaded():
        return candidates[:top_k]
    docs = [{"index": i, "content": str(c.get(content_key, ""))[:1500]}
            for i, c in enumerate(candidates)]
    sql = (
        f"SELECT t.idx, PREDICTION({RERANKER_MODEL} USING :q AS DATA1, t.content AS DATA2) "
        "  FROM JSON_TABLE(:docs, '$[*]' COLUMNS ("
        "         idx NUMBER PATH '$.index', content VARCHAR2(4000) PATH '$.content')) t "
        " ORDER BY 2 DESC FETCH FIRST :k ROWS ONLY"
    )
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, q=query, docs=json.dumps(docs), k=top_k)
            ranked = list(cur)
    except oracledb.DatabaseError:
        return candidates[:top_k]
    out = []
    for idx, score in ranked:
        if idx is None or int(idx) >= len(candidates):
            continue
        item = dict(candidates[int(idx)])
        item["rerank_score"] = float(score) if score is not None else 0.0
        out.append(item)
    return out


print(f"rerank ready (reranker loaded: {_reranker_loaded()})")

## 3.2 Implement `retrieve_knowledge`

Two stages:

1. Oversample by 4×: `memory_client.search(...)` with `max_results=k*4`.
2. Filter out `kind == "tool_output"`. Apply `kinds` filter if provided.
3. Rerank via `rerank(query, candidates, top_k=k, content_key="body")`.

> 📖 **See:** [Part 3 guide → TODO 2](../docs/part-3-retrieval.md#todo-5-implement-retrieve_knowledge)


In [ ]:
# TODO 2 — implement retrieve_knowledge(query, k=5, kinds=None) -> list[dict]

def retrieve_knowledge(query, k=5, kinds=None):
    # Defensive: LLMs sometimes pass kinds as a comma-separated string.
    if isinstance(kinds, str):
        kinds = [s.strip() for s in kinds.split(",") if s.strip()]
    if kinds == []:
        kinds = None

    cosine_fetch = k * 4
    hits = memory_client.search(
        query,
        user_id=USER_ID, agent_id=AGENT_ID,
        record_types=["memory"],
        max_results=cosine_fetch,
    )

    candidates = []
    for h in hits:
        meta = h.metadata or {}
        kind_value = meta.get("kind")
        if kind_value == "tool_output":
            continue
        if kinds is not None and (kind_value is None or kind_value not in kinds):
            continue
        candidates.append({
            "kind":     kind_value or "?",
            "subject":  meta.get("subject", ""),
            "body":     h.content,
            "metadata": meta,
            "distance": float(h.distance),
        })

    return rerank(query, candidates, top_k=k, content_key="body")


In [ ]:
# Hard-stop checkpoint: TODO 2
_hits = retrieve_knowledge("which table holds vessel capacity", k=3)
assert _hits, "❌ TODO 2: retrieve_knowledge returned no hits. Did you call rerank() at the end?"
assert all("body" in h for h in _hits), "❌ TODO 2: each hit must have a 'body' key."
print(f"✅ TODO 2 passed — retrieve_knowledge returns {len(_hits)} reranked hits.")
for h in _hits:
    print(f"  [{h['kind']:12s}] {h['subject'][:40]:40s}  {h['body'][:80]}")

## 3.3 Keyword search

`keyword_search_memories` uses the pre-created `CTXSYS.CONTEXT` index. This is the keyword leg of the hybrid retrieval in 3.4 — it scores by `SCORE(1)` over a `CONTAINS` predicate, and skips offloaded `tool_output` rows so they don't drown out semantic memories.


In [ ]:
MEMORY_TABLE = "eda_onnx_memory"

def keyword_search_memories(query, k=5):
    sql = f"""
        SELECT m.record_id, m.metadata,
               DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
               SCORE(1) AS score_txt
          FROM {MEMORY_TABLE} m
         WHERE CONTAINS(m.content, :kw, 1) > 0
           AND m.user_id = :u AND m.agent_id = :a
           AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
         ORDER BY SCORE(1) DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        cur.execute(sql, kw=query, u=USER_ID, a=AGENT_ID, k=k)
        rows = []
        for record_id, metadata, content, score in cur:
            meta = metadata or {}
            if hasattr(content, "read"):
                content = content.read()
            rows.append({"record_id": record_id,
                         "kind": meta.get("kind", "memory"),
                         "subject": meta.get("subject", ""),
                         "content": str(content or "")[:500],
                         "score_txt": float(score) if score is not None else 0.0})
    return rows

## 3.4 Hybrid RRF retrieval — `hybrid_rrf_search_memories`

**TODO 3 (in the student notebook).** Fuses the keyword leg from 3.3 with the vector leg from your Part 2 embedder in **one SQL statement** using Reciprocal Rank Fusion (RRF). Rows that appear in *both* lists outrank rows that appear in only one — and you never have to normalise scores between cosine distance and Oracle Text's BM25.

> 📖 **See:** [Part 3 guide → TODO 3](../docs/part-3-retrieval.md#todo-3-implement-hybrid_rrf_search_memories)


In [ ]:
def hybrid_rrf_search_memories(query, k=5, per_list=30, rrf_k=60):
    sql = f"""
        WITH q_emb AS (
            SELECT TO_VECTOR(VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), 384, FLOAT64) AS emb
              FROM dual),
        vec AS (
            SELECT m.record_id, DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content, m.metadata,
                   ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE)) AS r_vec
              FROM {MEMORY_TABLE} m
              JOIN eda_onnx_record_chunks c ON c.source_id = m.record_id
              CROSS JOIN q_emb
             WHERE m.user_id = :u AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY),
        txt AS (
            SELECT m.record_id, DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content, m.metadata,
                   ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
              FROM {MEMORY_TABLE} m
             WHERE CONTAINS(m.content, :kw, 1) > 0
               AND m.user_id = :u AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY)
        SELECT COALESCE(v.record_id, t.record_id) AS record_id,
               COALESCE(v.content, t.content)     AS content,
               COALESCE(v.metadata, t.metadata)   AS metadata,
               NVL(v.r_vec, 999999) AS r_vec,
               NVL(t.r_txt, 999999) AS r_txt,
               ( 1.0/(:rrf_k + NVL(v.r_vec, 999999))
               + 1.0/(:rrf_k + NVL(t.r_txt, 999999)) ) AS rrf_score
          FROM vec v
          FULL OUTER JOIN txt t ON v.record_id = t.record_id
         ORDER BY rrf_score DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        kw = f'"{query}"' if " " in query.strip() else query
        cur.execute(sql, q=query, kw=kw, u=USER_ID, a=AGENT_ID,
                    n=per_list, rrf_k=rrf_k, k=k)
        rows = []
        for rec_id, content, meta, r_vec, r_txt, rrf in cur:
            if hasattr(content, "read"):
                content = content.read()
            rows.append({"record_id": rec_id,
                         "kind": (meta or {}).get("kind", "memory"),
                         "subject": (meta or {}).get("subject", ""),
                         "content": str(content or "")[:500],
                         "r_vec": int(r_vec), "r_txt": int(r_txt),
                         "rrf_score": float(rrf)})
    return rows


print("keyword_search_memories + hybrid_rrf_search_memories ready")

## 3.5 Three-way probe (just run)

Same query through all three retrievers. Watch the `r_vec` / `r_txt` columns — rows in both lists get the highest combined RRF score.

In [ ]:
probe_q = "TEU 20-foot equivalent capacity unit for vessels"

print("=" * 80)
print(f"QUERY: {probe_q!r}")
print("=" * 80)

print("\n--- A) VECTOR + RERANK ---")
for h in retrieve_knowledge(probe_q, k=3):
    print(f"  [{h['kind']:12s}] {h['subject'][:40]:40s}  {h['body'][:100]}")

print("\n--- B) KEYWORD ONLY ---")
for h in keyword_search_memories(probe_q, k=3):
    print(f"  [{h['kind']:12s}] score={h['score_txt']:6.2f}  {h['subject'][:40]}")

print("\n--- C) HYBRID via RRF ---")
for h in hybrid_rrf_search_memories(probe_q, k=3):
    print(f"  rrf={h['rrf_score']:.4f}  r_vec={h['r_vec']:>3}  r_txt={h['r_txt']:>3}  {h['subject'][:40]}")

# Part 4 — DBFS Scratchpad

> 📖 **Guide:** [`docs/part-4-tools-and-skills.md`](../docs/part-6-tools-and-skills.md) (DBFS section)

*Block 4 of 11.* Parts 2 and 3 are the agent's long-term store: tables, vectors, indexes, search. Part 4 adds something different — a **short-term scratchpad** for mid-task work that doesn't deserve a row in the long-term store. Think of it like the analyst's whiteboard: SQL drafts, intermediate aggregations, running findings logs.


[Oracle DBFS](https://docs.oracle.com/en/database/oracle/oracle-database/26/adlob/database-filesystem-DBFS-intro.html) is a POSIX-like filesystem layered on SecureFile LOBs in a table. The agent sees files and directories; the database sees rows. The pre-built setup mounted a DBFS store at `/scratch` under the `AGENT` schema; here we wrap the PL/SQL `PUTPATH`/`GETPATH` calls behind a Python class so the rest of the harness can use `read` / `write` / `append` semantics.

Why a filesystem at all? Multi-step reasoning often involves drafting SQL, computing intermediate values, keeping a findings log — workflows that *want* file semantics rather than rows in a table. And because DBFS persists in Oracle SecureFiles, it shares the same backups, audit, and security model as the rest of the agent's state.

## 4.1 The `DBFS` Python wrapper

DBFS is exposed via PL/SQL primitives — `DBMS_DBFS_CONTENT.CREATEFILE`, `GETPATH`, `DELETEFILE`. None of that is ergonomic from Python. We wrap them behind `write()` / `read()` / `append()` / `list()` so the agent's tools (later in Part 6 — `tool_scratch_write`, `tool_scratch_read`) look like ordinary file I/O.

The DBFS store itself (`AGENT_SCRATCH` mounted at `/scratch`) was created by `app/scripts/bootstrap.py` — you don't run any DDL here, just instantiate the Python class once.

In [ ]:
DBFS_MOUNT = "/scratch"
DBFS_STORE = "AGENT_SCRATCH"


class DBFS:
    def __init__(self, conn, mount=DBFS_MOUNT):
        self.conn = conn
        self.mount = mount.rstrip("/")

    def _full(self, path):
        if not path.startswith("/"):
            path = "/" + path
        if path == self.mount or path.startswith(self.mount + "/"):
            return path
        return f"{self.mount}{path}"

    def write(self, path, content):
        data = content.encode("utf-8") if isinstance(content, str) else content
        full = self._full(path)
        delete_plsql = "BEGIN DBMS_DBFS_CONTENT.DELETEFILE(:p); EXCEPTION WHEN OTHERS THEN NULL; END;"
        create_plsql = (
            "DECLARE l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob BLOB := :b; "
            "BEGIN DBMS_DBFS_CONTENT.CREATEFILE("
            "  path => :p, properties => l_props, content => l_blob); COMMIT; END;"
        )
        with self.conn.cursor() as cur:
            cur.execute(delete_plsql, p=full)
            cur.setinputsizes(b=oracledb.DB_TYPE_BLOB)
            cur.execute(create_plsql, p=full, b=data)
        self.conn.commit()

    def append(self, path, content):
        new = content.encode("utf-8") if isinstance(content, str) else content
        try:
            existing = self.read(path).encode("utf-8")
            sep = b"" if existing.endswith(b"\n") or not existing else b"\n"
            self.write(path, existing + sep + new)
        except FileNotFoundError:
            self.write(path, new)

    def read(self, path):
        full = self._full(path)
        read_plsql = (
            "DECLARE l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob BLOB; l_item_type NUMBER; "
            "BEGIN DBMS_DBFS_CONTENT.GETPATH("
            "  path => :p, properties => l_props, content => l_blob, item_type => l_item_type); "
            "  :out := l_blob; END;"
        )
        try:
            with self.conn.cursor() as cur:
                out = cur.var(oracledb.DB_TYPE_BLOB)
                cur.execute(read_plsql, p=full, out=out)
                blob = out.getvalue()
            if blob is None:
                raise FileNotFoundError(full)
            return blob.read().decode("utf-8", errors="replace")
        except oracledb.DatabaseError as e:
            if e.args and e.args[0].code in (64002, 64007):
                raise FileNotFoundError(full) from e
            raise


scratch = DBFS(agent_conn)

# Smoke test
scratch.write("/scratch/hello.txt", "Hello from DBFS!")
print(scratch.read("/scratch/hello.txt"))

# Part 5 — Oracle MLE Compute Sandbox

> 📖 **Guide:** [`docs/part-5-mle.md`](../docs/part-5-mle.md)

*Block 5 of 11.* Parts 2–4 cover the agent's *state* (long-term memory + scratchpad). Part 5 adds the agent's *compute*. LLMs are unreliable at math — percentiles, weighted means, post-fetch reshaping — so we route those snippets through a deterministic engine that lives inside Oracle: the Multilingual Engine (MLE).


LLMs are unreliable at math. Percentiles, weighted means, post-fetch reshaping — anything quantitative — should run in a deterministic engine, not in the model's head. We route those snippets through Oracle's **Multilingual Engine (MLE)** — JavaScript that runs *inside* the Oracle process, called via `DBMS_MLE`.

Why MLE rather than a subprocess sandbox or a remote VM:

- **MLE is a language sandbox, not a privilege sandbox.** GraalVM blocks filesystem, network, and native code, but JS-side code inherits the caller's DB grants via `mle-js-oracledb`. - **No exfiltration channel by construction.** The polyglot engine doesn't expose filesystem, network, or native libs, so agent-authored code can't open sockets or read `~/.ssh/`.
- **Code runs next to the data.** Snippets that operate over fetched rows don't round-trip those rows back out.
- **Zero local install.** GraalVM ships *inside* the database. No Python / Java / GraalVM on the laptop running the harness.

## 5.1 The `exec_js` helper

`exec_js(code)` takes a JavaScript snippet, wraps it in a `try/catch` + `console.log` capture, evaluates it inside a fresh MLE context, and returns:

```python
{"stdout": "<captured console.log>", "stderr": "<error if thrown>", "ok": True/False}
```

Under the hood it uses three `DBMS_MLE` calls — `CREATE_CONTEXT`, `EVAL`, `IMPORT_FROM_MLE` — plus the `mle-js-bindings` module to pass a `result` value back to Python. The wrapper code is fiddly because MLE is fiddly; you'll rarely need to read it. Use `exec_js` as if it were `eval()` for a JS snippet, except it runs deterministically inside Oracle.

The smoke test below has the agent compute percentile statistics over an inline list of order totals — the kind of math you do NOT want an LLM doing in its head.

In [ ]:
def exec_js(code):
    wrapper = (
        '(function() {\n'
        '  let _stdout = ""; let _stderr = ""; let _ok = true;\n'
        '  const _origLog = console.log;\n'
        '  console.log = function() {\n'
        '    _stdout += Array.from(arguments).map(String).join(" ") + "\\n";\n'
        '  };\n'
        '  try {\n'
        + code + '\n'
        '  } catch (e) {\n'
        '    _stderr = String(e && e.message ? e.message : e) + "\\n" + (e && e.stack ? e.stack : "");\n'
        '    _ok = false;\n'
        '  } finally { console.log = _origLog; }\n'
        '  const bindings = require("mle-js-bindings");\n'
        '  bindings.exportValue("result", JSON.stringify({stdout: _stdout, stderr: _stderr, ok: _ok}));\n'
        '})();'
    )
    plsql = (
        "DECLARE l_ctx RAW(16); l_result CLOB; "
        "BEGIN l_ctx := DBMS_MLE.CREATE_CONTEXT(); "
        "  DBMS_MLE.EVAL(l_ctx, 'JAVASCRIPT', :source); "
        "  DBMS_MLE.IMPORT_FROM_MLE(l_ctx, 'result', l_result); "
        "  :result := l_result; "
        "  DBMS_MLE.DROP_CONTEXT(l_ctx); "
        "EXCEPTION WHEN OTHERS THEN "
        "  BEGIN DBMS_MLE.DROP_CONTEXT(l_ctx); EXCEPTION WHEN OTHERS THEN NULL; END; "
        "  RAISE; END;"
    )
    with agent_conn.cursor() as cur:
        out = cur.var(oracledb.DB_TYPE_CLOB)
        try:
            cur.setinputsizes(source=oracledb.DB_TYPE_CLOB)
            cur.execute(plsql, source=wrapper, result=out)
        except oracledb.DatabaseError as e:
            return {"stdout": "", "stderr": str(e), "ok": False}
    clob = out.getvalue()
    text = clob.read() if hasattr(clob, "read") else (clob or "")
    if not text:
        return {"stdout": "", "stderr": "MLE returned no output", "ok": False}
    try:
        return json.loads(text)
    except (ValueError, TypeError):
        return {"stdout": text, "stderr": "", "ok": True}


# Smoke test: compute percentiles + summary stats over a list of values
order_totals = [199, 4999, 12999, 599, 8999, 24999, 1499, 89999, 3499, 599, 19999, 11999]
snippet = f"""
const totals = {json.dumps(order_totals)}.slice().sort((a, b) => a - b);
const n = totals.length;
function pct(p) {{
    const k = (n - 1) * p;
    const f = Math.floor(k);
    return f === Math.min(f + 1, n - 1) ? totals[f] : totals[f] + (totals[Math.min(f + 1, n - 1)] - totals[f]) * (k - f);
}}
const sum = totals.reduce((a, b) => a + b, 0);
console.log("n=" + n + " mean=" + Math.floor(sum/n) + " p50=" + Math.round(pct(0.50)) + " p95=" + Math.round(pct(0.95)));
"""
print(exec_js(snippet)["stdout"], end="")

# Part 6 — Tools & Skills

> 📖 **Guide:** [`docs/part-6-tools-and-skills.md`](../docs/part-6-tools-and-skills.md)

> 🔧 **TODO in this part:** **TODO 4** — register `tool_run_sql`

*Block 6 of 11.* The agent now has memory (Part 2), retrieval (Part 3), a scratchpad (Part 4), and compute (Part 5) — but no way to *invoke* any of it. Part 6 wires up the dispatch layer. Tools are Python callables the model can request via OpenAI-style `tool_calls`; skills are prose playbooks the model reads on demand. Both are vector-indexed so per-turn prompt size stays flat as the registry grows.


A **tool** is two things:

1. A Python callable. Its name, docstring, type hints, and signature defaults *are* its public spec.
2. A row in the `toolbox` table — name, description, parameters JSON, embedding. We retrieve top-k tools per turn by vector similarity over the user query, so the registry can grow past 30 tools without bloating per-turn cost.

A **skill** is a prose markdown playbook with the same retrieval pattern but a different consumption model — the model reads it on demand via `load_skill(name)`.

![Toolbox flow — registration vs per-turn retrieval](../images/cover-toolbox-flow.png)

The `toolbox` and `skillbox` tables are already created and seeded (skillbox has ~155 skills from [`oracle/skills`](https://github.com/oracle/skills)). Here we define the `@register` decorator, the `TOOLS` registry, and the tools themselves.

## 6.1 The `@register` decorator

One decorator does three things:

1. **Introspects the function** — `__name__`, `__doc__`, type hints, default values — and builds an OpenAI-compatible JSON schema (what the LLM sees as the tool's signature).
2. **Registers it** in the in-process `TOOLS` dict so the agent loop can dispatch by name.
3. **Embeds the description** in SQL via `VECTOR_EMBEDDING` and writes it to the `toolbox` table — making the tool retrievable by cosine similarity to the user's query.

The third step is what lets the registry grow past 30 tools without bloating per-turn prompts: `retrieve_tools(query, k=6)` does cosine search over `toolbox.embedding` (HNSW-indexed) + cross-encoder rerank, then merges in a small set of always-on tools (`run_sql`, `search_knowledge`, `remember`, `exec_js`, `load_skill`). The model only sees the relevant subset for each turn.

> **HNSW** = Hierarchical Navigable Small World. Oracle's vector index type for fast approximate nearest-neighbour search. Lives in the `vector_memory_size` SGA pool (set to 512M by `bootstrap.py`). Lookups grow as `log(N)` rather than `N`.

In [ ]:
import inspect, typing

TOOLS = {}                                          # name -> (callable, openai_schema)
ALWAYS_ON_TOOLS = {"search_knowledge", "run_sql", "remember", "exec_js"}

_PRIMS = {int: "integer", float: "number", bool: "boolean", str: "string"}


def _hint_to_json(hint):
    origin = typing.get_origin(hint)
    if hint in _PRIMS:
        return {"type": _PRIMS[hint]}
    if origin in (list, typing.List):
        args = typing.get_args(hint) or (str,)
        return {"type": "array", "items": _hint_to_json(args[0])}
    if origin in (dict, typing.Dict):
        return {"type": "object"}
    if origin is typing.Union:
        non_none = [a for a in typing.get_args(hint) if a is not type(None)]
        if len(non_none) == 1:
            return _hint_to_json(non_none[0])
    return {"type": "string"}


def _build_schema(fn):
    raw_name = fn.__name__
    name = raw_name[5:] if raw_name.startswith("tool_") else raw_name
    description = (inspect.getdoc(fn) or "").strip()
    if not description:
        raise ValueError(f"tool {name!r} has no docstring; @register needs one for retrieval")
    sig = inspect.signature(fn)
    hints = typing.get_type_hints(fn)
    properties, required = {}, []
    for pname, param in sig.parameters.items():
        prop = _hint_to_json(hints.get(pname, str))
        if param.default is not inspect.Parameter.empty and param.default is not None:
            prop["default"] = param.default
        else:
            required.append(pname)
        properties[pname] = prop
    parameters = {"type": "object", "properties": properties, "required": required}
    return name, description, parameters, {
        "type": "function",
        "function": {"name": name, "description": description, "parameters": parameters},
    }



In [ ]:
def register(fn):
    name, description, parameters, openai_schema = _build_schema(fn)
    TOOLS[name] = (fn, openai_schema)
    arg_text = " ".join(parameters["properties"].keys())
    embed_text = f"{name}: {description}\nargs: {arg_text}"
    with agent_conn.cursor() as cur:
        cur.execute(
            "MERGE INTO toolbox t USING (SELECT :tn AS n FROM dual) s ON (t.name = s.n) "
            "WHEN MATCHED THEN UPDATE SET description = :td, parameters = :tp, "
            f" embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
            " updated_at = CURRENT_TIMESTAMP "
            "WHEN NOT MATCHED THEN INSERT (name, description, parameters, embedding) "
            f" VALUES (:tn, :td, :tp, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA))",
            tn=name, td=description, tp=json.dumps(parameters), etext=embed_text,
        )
    agent_conn.commit()
    return fn

In [ ]:
def retrieve_tools(query, k=6):
    cosine_fetch = k * 4
    rows = []
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, description FROM toolbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY", q=query, k=cosine_fetch)
        for name, desc in cur:
            desc_text = desc.read() if hasattr(desc, "read") else str(desc or "")
            rows.append({"name": name, "content": desc_text})
    ranked = rerank(query, rows, top_k=k, content_key="content")
    schemas = {}
    for r in ranked:
        if r["name"] in TOOLS:
            schemas[r["name"]] = TOOLS[r["name"]][1]
    for name in ALWAYS_ON_TOOLS:
        if name in TOOLS:
            schemas[name] = TOOLS[name][1]
    return list(schemas.values())

## 6.2 Register `tool_run_sql`

The agent's primary way to query live data. Must be **read-only**: `SELECT` and `WITH` only.

The function should:
1. Reject statements that don't match `_READ_ONLY`.
2. Execute the SQL on `agent_conn`.
3. Return up to `max_rows` rows as JSON: `{"columns": [...], "rows": [...], "row_count": N}`.
4. Handle CLOB cells: `v.read() if hasattr(v, "read") else v`.
5. Return `{"error": "..."}` on any database error.

The docstring is what the LLM sees as the tool description — make it descriptive.

> 📖 **See:** [Part 6 guide → TODO 4](../docs/part-6-tools-and-skills.md#todo-9-register-tool_run_sql)


In [ ]:
_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


# TODO 4 — decorate with @register and implement.

@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    if not _READ_ONLY.match(sql.strip()):
        return json.dumps({"error": "only SELECT / WITH statements are allowed in run_sql"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = []
            for i, r in enumerate(cur):
                if i >= max_rows:
                    break
                rows.append([(v.read() if hasattr(v, "read") else v) for v in r])
        return json.dumps({"columns": cols, "rows": rows, "row_count": len(rows)}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


In [ ]:
# Hard-stop checkpoint: TODO 4
assert "run_sql" in TOOLS, "❌ TODO 4: tool_run_sql not registered. Did you @register it?"
_out = json.loads(tool_run_sql("SELECT 1 AS one FROM dual"))
assert _out.get("row_count") == 1, "❌ TODO 4: SELECT 1 didn't return one row."
_bad = json.loads(tool_run_sql("DROP TABLE x"))
assert "error" in _bad, "❌ TODO 4: read-only guard let a DROP through!"
print("✅ TODO 4 passed — tool_run_sql registered and read-only enforced.")

## 6.3 The rest of the toolset

Pre-built tools that round out the harness: `scan_database`, `search_knowledge`, `exec_js`, `scratch_*`, `remember`.

In [ ]:
@register
def tool_scan_database(owner: str) -> str:
    """Scan the specified schema of the target Oracle AI Database and update institutional knowledge.
    Run this when the user asks about a schema you have never seen or when you suspect the knowledge store is stale.
    """
    return json.dumps(run_scan(agent_conn, owner=owner))


@register
def tool_search_knowledge(query: str, k: int = 5, kinds: list[str] | None = None) -> str:
    """Search institutional knowledge by semantic similarity.
    Use this BEFORE running SQL to discover which tables and columns are relevant.
    `kinds` is an optional filter: table, column, relationship, query_pattern, correction.
    """
    hits = retrieve_knowledge(query, k=k, kinds=kinds)
    for h in hits:
        h["body"] = h["body"][:500]
    return json.dumps(hits)


@register
def tool_exec_js(code: str) -> str:
    """Execute JavaScript inside Oracle MLE (no filesystem, no network).
    Good for arithmetic, string formatting, JSON reshaping, simple aggregations.
    `console.log(...)` output comes back as `stdout`.
    """
    return json.dumps(exec_js(code))


@register
def tool_scratch_write(path: str, content: str) -> str:
    """Write a short-term note to the DBFS scratchpad. REPLACES any existing file at path."""
    scratch.write(path, content)
    return json.dumps({"ok": True, "path": path, "bytes": len(content)})


@register
def tool_scratch_append(path: str, content: str) -> str:
    """Append text to the end of a DBFS scratchpad file (or create it).
    Use for running findings logs that should grow without overwriting prior entries.
    """
    scratch.append(path, content)
    return json.dumps({"ok": True, "path": path, "appended_bytes": len(content)})


@register
def tool_scratch_read(path: str) -> str:
    """Read from the DBFS scratchpad."""
    try:
        return json.dumps({"content": scratch.read(path)})
    except FileNotFoundError:
        return json.dumps({"error": f"not found: {path}"})


@register
def tool_remember(subject: str, body: str, kind: str = "correction") -> str:
    """Persist a correction or learning into institutional knowledge so future turns benefit.
    Use when the user corrects you, or when you discover a non-obvious fact that future retrievals should surface.
    """
    fact = Fact(kind=kind, subject=subject, body=body, metadata={"source": "agent_remember"})
    new, updated, _ = write_facts([fact])
    return json.dumps({"ok": True, "new": new, "updated": updated})


print(f"registered {len(TOOLS)} tools: {sorted(TOOLS)}")

## 6.4 Skill manifest

The skillbox is pre-populated. `build_skill_manifest(query, k=3)` returns the top-k skill names + descriptions, formatted as a one-line-per-skill block prepended to every turn's context. The model can `load_skill(name)` to retrieve the full body.

In [ ]:
@register
def tool_load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the system prompt's manifest lists a skill relevant to the current task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    with agent_conn.cursor() as cur:
        cur.execute("SELECT description, body, source_url, category FROM skillbox WHERE name = :n",
                    n=name)
        row = cur.fetchone()
    if not row:
        return json.dumps({"error": f"no skill named {name!r}"})
    desc, body, url, category = row
    body_text = body.read() if hasattr(body, "read") else str(body or "")
    return json.dumps({"name": name, "category": category, "description": desc,
                       "source_url": url, "body": body_text})


@register
def tool_list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns top-k skills (name + description)."""
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, category, description FROM skillbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY", q=query, k=k)
        hits = [{"name": n, "category": c, "description": d} for n, c, d in cur]
    return json.dumps(hits)


ALWAYS_ON_TOOLS.add("load_skill")


def build_skill_manifest(query, k=3):
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT name, description FROM skillbox "
                f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
                " FETCH FIRST :k ROWS ONLY", q=query, k=k)
            rows = list(cur)
    except oracledb.DatabaseError:
        return ""
    if not rows:
        return ""
    lines = [f"  - {n} — {(d.read() if hasattr(d, 'read') else d)[:240]}" for n, d in rows]
    return ("Available skills (call load_skill(name) to read the full guide and follow it):\n"
            + "\n".join(lines) + "\n\n")


print(f"registered: load_skill, list_skills (TOOLS total: {len(TOOLS)})")

# Part 7 — The Agent Loop

> 📖 **Guide:** [`docs/part-7-agent-loop.md`](../docs/part-7-agent-loop.md)

> 🔧 **TODO in this part:** **TODO 5** — `agent_turn`

*Block 7 of 11. **The keystone.*** Parts 2–6 are plumbing — each one a primitive sitting on its own. Part 7 is the loop that ties them together: assemble context (skills + memory + retrieved knowledge), call the LLM with the retrieved tools, dispatch any tool the model emits, append the result, repeat. Roughly 90 lines of Python — read this one twice. Everything in Parts 8–11 *extends* this loop.


Everything we've built so far is plumbing. **This** is the agent. Read this section twice.

```
build_context → call LLM with retrieved tools → if tool_calls: dispatch  else: final → log
```

## 7.1 `SYSTEM_PROMPT` + `build_context` + `log_message`

Three pieces the agent loop relies on:

1. **`SYSTEM_PROMPT`** — the agent's job description. Read it carefully; every rule there shapes a real choice the model makes per turn (e.g. *"For numeric work, you MUST fetch with `run_sql` then call `exec_js`"* — without this rule GPT-class models do arithmetic in their head and get it wrong).
2. **`get_thread(thread_id)`** — a cache of OAMP thread objects. Threads hold per-conversation state: messages, the rolling LLM-written summary, the context card.
3. **`build_context(thread_id, user_query)`** — assembles three layers into one user message: skill manifest (Part 6) → OAMP context card → retrieved knowledge (Part 3) → the user's question. The model sees one cohesive user turn, not four concatenated blocks.
4. **`log_message`** — fire-and-forget persistence of each turn to OAMP. We submit to a thread pool so the agent loop never blocks on extraction-LLM work.

These are the building blocks. The actual loop is your TODO 5.

In [ ]:
SYSTEM_PROMPT = (
    "You are an Enterprise Data Agent operating against an Oracle AI Database.\n"
    "\n"
    "Your job is to answer natural-language questions about the data by reasoning over:\n"
    "  - the institutional knowledge you have accumulated about the database;\n"
    "  - the live database, via read-only SQL when you need runtime facts;\n"
    "  - a scratchpad for intermediate notes;\n"
    "  - an in-database JavaScript sandbox via Oracle MLE (exec_js) for computation.\n"
    "\n"
    "How to work:\n"
    "  1. ALWAYS call search_knowledge first with a paraphrase of the user's question.\n"
    "  2. If the user references a schema you have no facts about, call scan_database, then search again.\n"
    "  3. Keep SQL read-only. Use run_sql only; never attempt DDL or DML through it.\n"
    "  4. For numeric work over fetched rows - mean / median / percentile / unit conversion -\n"
    "     you MUST fetch with run_sql and then call exec_js to compute. Do not compute statistics\n"
    "     in your head, and do not lean only on SQL aggregation when JS is more honest about the math.\n"
    "  5. For non-trivial SQL, draft to /scratch/<task>.sql via scratch_write, then scratch_read\n"
    "     before passing to run_sql. The scratchpad persists across tool calls and turns.\n"
    "  6. When you discover a non-obvious fact or the user corrects you, call remember\n"
    "     so future turns benefit.\n"
    "  7. Prefer short, direct answers. Quote table and column names verbatim.\n"
    "  8. If a tool fails, read the error, adjust, and try once more - don't spin.\n"
    "\n"
    "Never fabricate a table or column. If you're unsure, say so and propose a scan."
)


THREADS = {}


In [ ]:
def get_thread(thread_id):
    if thread_id in THREADS:
        return THREADS[thread_id]
    try:
        thread = memory_client.get_thread(thread_id)
    except Exception:
        thread = memory_client.create_thread(
            thread_id=thread_id, user_id=USER_ID, agent_id=AGENT_ID,
            memory_extraction_frequency=2, memory_extraction_window=4,
            enable_context_summary=True, context_summary_update_frequency=4,
        )
    THREADS[thread_id] = thread
    return thread


In [ ]:

def build_context(thread_id, user_query, k_knowledge=3):
    parts = []
    try:
        manifest = build_skill_manifest(user_query, k=3)
    except NameError:
        manifest = ""
    if manifest:
        parts.append(manifest.rstrip())

    thread = get_thread(thread_id)
    card = thread.get_context_card()
    card_text = str(card) if card else ""
    if card_text:
        parts.append("## Memory context (from OAMP)")
        parts.append(card_text)

    hits = retrieve_knowledge(user_query, k=k_knowledge)
    if hits:
        parts.append("\n## Institutional knowledge (top matches)")
        for h in hits:
            parts.append(f"- ({h['kind']}) {h['subject']} - {h['body'][:280]}")

    parts.append("\n## User question")
    parts.append(user_query)
    return "\n".join(parts)

In [ ]:
_LOG_EXECUTOR = concurrent.futures.ThreadPoolExecutor(max_workers=4, thread_name_prefix="oamp-log")
_LOG_PENDING = []

In [ ]:


def log_message(thread_id, role, content):
    from oracleagentmemory.apis.thread import Message
    def _do_log():
        get_thread(thread_id).add_messages([Message(role=role, content=content)])
    still = [f for f in _LOG_PENDING if not f.done()]
    _LOG_PENDING[:] = still
    _LOG_PENDING.append(_LOG_EXECUTOR.submit(_do_log))


print("SYSTEM_PROMPT + get_thread + build_context + log_message ready")

## 7.2 Implement `agent_turn`

The heart of the harness. Loop: build context → call LLM with retrieved tools → if `tool_calls` dispatch and append results → else break with final answer. Guard with iteration cap and wall-clock budget. If both run out without a final answer, ask one more time without tools (forced final).

> 📖 **See:** [Part 7 guide → TODO 5](../docs/part-7-agent-loop.md#todo-5-implement-agent_turn)


In [ ]:
# TODO 5 — implement agent_turn(user_query, thread_id, max_iterations, budget_seconds, verbose)
# 📖 docs/part-7-agent-loop.md

def agent_turn(user_query, thread_id="default", max_iterations=8, budget_seconds=360.0, verbose=True):
    started = time.time()
    log_message(thread_id, "user", user_query)

    context = build_context(thread_id, user_query)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        # Echo assistant tool_calls back so each tool result has its matching call.
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            if name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)
    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240]},
        )
    except Exception:
        pass

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


In [ ]:
# Hard-stop checkpoint: TODO 5
assert callable(agent_turn), "❌ TODO 5: agent_turn not defined."
import inspect as _ins
_sig = _ins.signature(agent_turn)
for _p in ("user_query", "thread_id", "max_iterations", "budget_seconds"):
    assert _p in _sig.parameters, f"❌ TODO 5: agent_turn missing parameter {_p!r}."
print("✅ TODO 5 passed — agent_turn signature looks right. Demos below will exercise it.")

## 7.3 Three-turn demo (just run)

Three turns on the same thread, each exercising a different harness component:

1. Discovery (forces `search_knowledge`).
2. Live data (forces `run_sql`).
3. Correction + persistence (forces `remember`).

In [ ]:
thread = "demo-session-1"

q1 = "What's in the SUPPLYCHAIN schema? Briefly — list the entities and how they relate."
print("USER:", q1)
print("\nASSISTANT:", agent_turn(q1, thread_id=thread))

In [ ]:
q2 = "How many active voyages does each carrier currently have? Show me a small table sorted by count desc."
print("USER:", q2)
print("\nASSISTANT:", agent_turn(q2, thread_id=thread))

In [ ]:
q3 = ("Important: in the SUPPLYCHAIN schema, vessels.capacity_teu is always in 20-foot equivalent units (TEU), "
      "never tons. And cargo_items.unit_value_cents is always USD CENTS, never dollars. "
      "Save EACH as a separate 'correction' memory by calling remember BEFORE you respond, "
      "then confirm with the memory IDs.")
print("USER:", q3)
print("\nASSISTANT:", agent_turn(q3, thread_id=thread))

# Part 9 — JSON Relational Duality Views

> 📖 **Guide:** [`docs/part-9-duality-views.md`](../docs/part-9-duality-views.md)

> 🔧 **TODO in this part:** **TODO 7** — register `tool_get_document`

*Block 9 of 11.* Parts 1–8 give the agent a working SQL interface to the relational schema. Part 9 adds a second projection of the same data: **document-shaped JSON reads**, where one row in `voyages` materialises as a nested document with its vessel, carrier, ports, containers, and cargo. Same trust boundary (Part 8's row policy applies *inside* the view); easier for the model to reason over.


A **duality view** is a JSON projection over a set of tables joined by PK/FK/UK relationships. The same row in `voyages` is accessible as a relational tuple AND as a nested JSON document.

`voyage_dv` and `vessel_dv` are pre-created on `SUPPLYCHAIN`. Both are read-only — no `WITH UPDATE`, so DML is rejected by the kernel. The Part 8 row policy applies *inside* the duality view too.

![JSON Relational Duality — three lenses, one source of truth](../images/cover-duality-view.png)

## 9.1 Register `tool_get_document`

Read one full document from a duality view by primary key. Whitelist views to `{"voyage_dv", "vessel_dv"}`. Bind `key` as int when it parses as a digit, else as string.

> 📖 **See:** [Part 9 guide → TODO 7](../docs/part-9-duality-views.md#todo-7-register-tool_get_document)


In [ ]:
# TODO 7 — register tool_get_document with @register.

@register
def tool_get_document(view: str, key: str) -> str:
    """Read one full document from a JSON Relational Duality View by primary key.
    Use this instead of writing JOINs whenever you need the full shape of an entity
    (voyage with vessel/carrier/ports/containers/cargo, or vessel with carrier/position).
    `view` must be one of: voyage_dv, vessel_dv.
    `key` is the value of the document _id (numeric voyage_id or vessel_id, as a string).
    """
    allowed = {"voyage_dv", "vessel_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                f"SELECT JSON_SERIALIZE(data PRETTY) FROM {DEMO_USER}.{view} "
                f" WHERE JSON_VALUE(data, '$._id') = :k",
                k=int(key) if str(key).isdigit() else key,
            )
            row = cur.fetchone()
        if not row:
            return json.dumps({"error": f"no document with _id={key} in {view}"})
        body = row[0].read() if hasattr(row[0], "read") else str(row[0])
        return body
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})


# Pre-built companion tool

@register
def tool_query_documents(view: str, where: str = "1=1", max_rows: int = 10) -> str:
    """Filter a JSON Relational Duality View with a SQL predicate.
    `view` must be one of: voyage_dv, vessel_dv.
    `where` is a SQL boolean expression on the root table's columns (default '1=1').
    """
    allowed = {"voyage_dv", "vessel_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    sql = f"SELECT JSON_SERIALIZE(data) FROM {DEMO_USER}.{view} WHERE {where} FETCH FIRST :n ROWS ONLY"
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, n=max_rows)
            docs = [(r[0].read() if hasattr(r[0], "read") else str(r[0])) for r in cur]
        return json.dumps({"count": len(docs), "documents": [json.loads(d) for d in docs]}, default=str)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}", "sql": sql})


In [ ]:
# Hard-stop checkpoint: TODO 7
assert "get_document" in TOOLS, "❌ TODO 7: tool_get_document not registered."
_out = tool_get_document(view="voyage_dv", key="1")
try:
    _doc = json.loads(_out)
except Exception:
    raise AssertionError(
        f"❌ TODO 7: tool returned non-JSON: {_out[:200]!r}")
# A known-good key MUST come back as a real document, not as an
# {"error": ...} payload. If the duality view is mis-aliased, the SQL
# will fail with ORA-00904 and the assert below catches it.
assert "error" not in _doc, (
    f"❌ TODO 7: tool returned an error for voyage_id=1: {_doc.get('error')!r}. "
    f"Check the SQL — the duality view's JSON column must be named DATA "
    f"(see app/backend/db/seed_supplychain.py for the view DDL).")
assert _doc.get("_id") == 1, (
    f"❌ TODO 7: expected _id=1 in the returned document, got {_doc.get('_id')!r}")
print("✅ TODO 7 passed — tool_get_document fetches a real document.")
print(_out[:400])

## 9.2 Duality view demo (just run)

The agent should reach for `get_document("voyage_dv", "7")` and get the full nested document back in one tool call — no manual JOINs.

In [ ]:
q_dv = (
    "Give me a complete picture of voyage_id 7: which carrier and vessel is running it, "
    "what ports it's between, what's loaded on it (containers and cargo items), and the "
    "vessel's current position. Use the duality view if you have one — that's why we built it."
)
print(agent_turn(q_dv, thread_id="demo-dv-1"))

# Part 11 — Tool-Output Offload

> 📖 **Guide:** [`docs/part-11-tool-output-offload.md`](../docs/part-11-tool-output-offload.md)

> 🔧 **TODOs in this part (2):** **TODO 8** — `log_tool`; **TODO 9** — register `tool_fetch_tool_output`

*Block 11 of 11. **The last brick.*** The Part 7 agent loop inlines every tool result verbatim into the next LLM call. That's fine for short outputs but blows the context window on a 50-row `run_sql` or a multi-KB skill body. Part 11 redefines `agent_turn` once more: full output → OAMP memory keyed by `tool_call_id`; inline → a 600-byte preview with a recovery hint. Bandwidth follows attention.


Part 7's `agent_turn` inlines every tool result verbatim. Fine for short outputs, but **blows the context window** on a 50-row `run_sql`, a multi-KB skill body, or a long `exec_js` log.

Three pieces glued together:

1. `log_tool` — every dispatch persists the full output as an OAMP memory tagged `kind=tool_output` with the LLM's `tool_call_id`.
2. **Truncation marker** — outputs over 600 bytes are replaced with a compact preview ending in `...[+N bytes. full output: fetch_tool_output(tool_call_id='call_…')]`.
3. `fetch_tool_output(tool_call_id)` — your TODO 9 — the agent-side retrieval tool.

## 11.1 `log_tool` — the write side of offload

**TODO 8 (in the student notebook).** Every dispatch persists the **full** tool output as an OAMP memory tagged `kind=tool_output` with the LLM's `tool_call_id`. The agent loop later inlines only a 600-byte preview and reaches for `fetch_tool_output(tool_call_id)` (TODO 9) when it needs the missing bytes.

> 📖 **See:** [Part 11 guide → TODO 8](../docs/part-11-tool-output-offload.md#todo-8-implement-log_tool)


In [ ]:
def log_tool(thread_id, tool_call_id, tool_name, tool_args, tool_output):
    memory_client.add_memory(
        tool_output,
        user_id=USER_ID, agent_id=AGENT_ID,
        thread_id=thread_id,
        metadata={
            "kind": "tool_output",
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "tool_args": json.dumps(tool_args),
        },
    )
print("log_tool ready")

## 11.2 Register `tool_fetch_tool_output`

Use `memory_client._store.list("memory", ..., metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id}, limit=1)`. Return JSON with `tool_name` / `tool_args` / `tool_output` if found, or `{"error": "..."}` if no record matches.

> 📖 **See:** [Part 11 guide → TODO 9](../docs/part-11-tool-output-offload.md#todo-9-register-tool_fetch_tool_output)


In [ ]:
# TODO 9 — register tool_fetch_tool_output with @register.

@register
def tool_fetch_tool_output(tool_call_id: str) -> str:
    """Retrieve the full, untruncated output of a previous tool call.
    Use this when a prior tool result in your context was truncated with
    '...[+N bytes. full output: fetch_tool_output(tool_call_id=...)]' and you need
    the missing bytes to answer.
    """
    records = memory_client._store.list(
        "memory",
        user_id=USER_ID, agent_id=AGENT_ID,
        metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id},
        limit=1,
    )
    if not records:
        return json.dumps({"error": f"no tool call with id {tool_call_id}"})
    r = records[0]
    meta = r.metadata or {}
    return json.dumps({
        "tool_name":   meta.get("tool_name"),
        "tool_args":   meta.get("tool_args"),
        "tool_output": r.content,
    })


In [ ]:
# Hard-stop checkpoint: TODO 9
assert "fetch_tool_output" in TOOLS, "❌ TODO 9: tool_fetch_tool_output not registered."
print("✅ TODO 9 passed — tool_fetch_tool_output registered.")

## 11.3 Redefine `agent_turn` with offload + truncation marker

This is the second (and final) definition of `agent_turn` in the workshop:

| Where | What it does |
|---|---|
| **Part 7** (your TODO 5) | The minimal loop. Tool outputs inlined verbatim. |
| **Part 11** (here) | Replaces the dispatch tail: full output → OAMP via `log_tool`; inline → 600-byte preview with recovery hint. |

After this cell runs, every later call to `agent_turn` uses the offload-version. To revert to the minimal Part 7 version, re-run the Part 7 cell.

In [ ]:
_prior_agent_turn_v2 = agent_turn


def agent_turn(user_query, thread_id="default", max_iterations=8, budget_seconds=360.0,
               verbose=True):
    """Agent loop with tool-output offload + truncation marker."""

    started = time.time()
    log_message(thread_id, "user", user_query)
    context = build_context(thread_id, user_query)
    messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            final = msg.content or ""
            break

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            if name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})

            # Offload full bytes; inline a compact preview with a recovery hint.
            log_tool(thread_id, tc.id, name, args, output)
            if len(output) <= 600:
                preview = output
            else:
                preview = (
                    output[:600] +
                    f" ...[+{len(output)-600} bytes. "
                    f"full output: fetch_tool_output(tool_call_id='{tc.id}')]"
                )
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": preview})

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)
    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception:
        pass
    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


print("agent_turn redefined: offload + truncation marker active.")

# Closing thoughts — and the running app

You've built the whole harness:

- **Long-term memory** via OAMP — durable facts, threads, context cards.
- **Hybrid retrieval** — cosine + cross-encoder rerank + Oracle Text fused via RRF in one SQL.
- **DBFS scratchpad** — file semantics on top of SecureFile LOBs, persisted across turns.
- **Oracle MLE compute** — JavaScript sandbox inside the database for deterministic math.
- **Vector-indexed `toolbox` and `skillbox`** — registry retrieval that scales past 30 tools.
- **The agent loop** — ~90 lines that orchestrate everything.
- **JSON Relational Duality Views** — the same row served as a tuple and a document.
- **Tool-output offload** — preview inline, full bytes recoverable by `tool_call_id`.

## See it running — open the app

The Codespace started a Flask + React deployment of this same harness on first launch. Open it now:

> **http://localhost:3000** (auto-forwarded; check the **PORTS** tab if it didn't open in a preview)

Same Oracle, same OAMP store, same `toolbox` and `skillbox` you populated in this notebook. What's different:

- A real chat UI, with each tool dispatch shown as a separate bubble.
- A live **memory pane** on the right — top semantic memories, recent tool outputs, skill manifest, token usage.
- An **identity selector** in the header — pick a persona and the same SQL returns different rows.
- A **3D globe** the agent can drive via the `focus_world` tool.
- Live web/news access via `search_tavily` (set `TAVILY_API_KEY` to enable).

For the full app architecture, read [`app/README.md`](../app/README.md). For everything that's pre-built in Oracle (DDL), see `app/scripts/bootstrap.py`, `seed.py`, and `setup_advanced.py`.

## Want to see *how* it's all set up?

The setup-stripped notebook above is meant for learning the harness pattern. If you want the version that *includes* the Oracle DDL, bootstrap, SUPPLYCHAIN seed, etc. (everything `app/scripts/*` would run for you), open:

> [`workshop/notebook_complete_with_setup_code.ipynb`](notebook_complete_with_setup_code.ipynb)

That's the full source — useful when you want to deploy this against an Oracle that *isn't* the workshop Codespace.